# Secure city network: design, risk, and attacks today

*Secure the Grid · Step 3 · team notebook*

The track asks us to design a metro-scale utility network: where to use quantum keys, where to use post-quantum cryptography, what assumptions we make, how we roll it out, and what a board would get for the cost.

We designed a **small illustrative network** (2 control centers, 8 substations, 16 fiber links) placed around Chattanooga for realistic distances. It is **not** EPB’s real layout. Numbers live in `data/` next to this folder; helpers we wrote are in `secure_grid/`.

In [ ]:
# Use the qBraid-SDK (v0.12.2) kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: this one has no " + " or ".join(missing) + ". Click the kernel name "
                      "at the top right of the notebook, pick qBraid-SDK (v0.12.2), then run this cell again.")
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

from pathlib import Path
import sys
_here = Path.cwd().resolve()
for _cand in (_here, _here.parent):
    if (_cand / "secure_grid").is_dir():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

import pandas as pd
import matplotlib.pyplot as plt
from secure_grid.network import load, link_table, qkd_check, assumption
from secure_grid import attacks, risk
import numpy as np

net = load()
defenses = attacks.load_defenses()
pd.set_option("display.max_colwidth", 70)
print(f"{len(net['nodes'])} sites, {len(net['links'])} links.")

## Part 1 — architecture

**Objective:** show where we use QKD and where we use post-quantum crypto, and why.

Every link gets three layers:

1. **Key** — ML-KEM (hybrid) on every link; QKD *plus* ML-KEM on the backbone and the spokes to the hospital, water plant, and airport.
2. **Authentication** — post-quantum tags on commands, with counters so an old command cannot be replayed.
3. **Detection** — QKD’s error alarm on those critical links; failed-authentication alarms everywhere.

Remote access needs a second factor. Clocks are protected (GPS spoofing breaks timestamps and replay defenses). Firmware uses hash-based signatures with a version counter so old signed images cannot be rolled back.

In [ ]:
print("Sites")
print(pd.DataFrame(net["nodes"])[["id", "name", "type", "cip_impact", "critical_load"]].fillna("").to_string(index=False))
print("\nLinks and QKD reach")
links = pd.DataFrame(link_table(net))
print(links[["link", "role", "criticality", "fiber_km", "loss_db", "direct_qkd", "key_rate_bps"]].round(1).to_string(index=False))
max_km = qkd_check(net, 0)["max_direct_km"]
print(f"\nWith our assumptions, one QKD pair reaches about {max_km:.0f} km. "
      f"Longest metro link here: {links.fiber_km.max():.0f} km.")
for regional_km in (120, 250):
    c = qkd_check(net, regional_km)
    print(f"A {regional_km} km regional link would need {c['trusted_nodes_needed']} trusted node(s).")

## Part 2 — assumptions

- Fiber is ~30% longer than a straight line (roads and poles). Loss ~0.2 dB/km plus connectors; QKD loss budget ~18 dB (so ~75 km max). These are labeled in `data/city_network.json`; the EPB field test (Evans et al., 2021) is the citation we would harden before a real board pitch.
- Cost lines for QKD pairs and PQC upgrades are still placeholders.
- Inside the city, **distance does not rule QKD out**. Cost, dedicated fiber, and endpoints do. Past metro distances we need trusted nodes, and that is where PQC wins outright.

## Today vs our design

Same network, two setups in `data/defenses.json`:

- **Today:** an illustrative typical utility — a few older substations still send DNP3 in the clear; VPN/TLS elsewhere use RSA/ECC; no MFA on a vendor remote account; clocks trust plain GPS; firmware can be rolled back.
- **Our design:** the three layers above.

We ran seven attacks as yes/no rules on every link or site they apply to. Six of them need **no quantum computer**.

In [ ]:
rows = []
for link in net["links"]:
    today = attacks.link_protection("today", defenses["today"], link)
    design = attacks.link_protection("design", defenses["design"], link)
    rows.append({"link": attacks.link_id(link), "role": link["role"],
                 "today key": today["key"], "today auth": today["authenticated"],
                 "design key": design["key"], "tap detection": design["tap_detection"]})
print(pd.DataFrame(rows).to_string(index=False))

board = pd.DataFrame(attacks.scoreboard(attacks.evaluate(net, defenses)))
print("\nScoreboard (targets where the attack succeeds)")
print(board.drop(columns=["quantum's role"]).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
y = list(range(len(board)))[::-1]
today = board["succeeds today"] / board["targets"]
design = board["succeeds on our design"] / board["targets"]
ax.barh([v + 0.18 for v in y], today, height=0.35, color="tab:red", label="succeeds today")
ax.barh([v - 0.18 for v in y], design, height=0.35, color="tab:green", label="succeeds on our design")
for v, (ok, n) in zip(y, zip(board["succeeds today"], board["targets"])):
    ax.text(ok / n + 0.02, v + 0.18, f"{ok}/{n}", va="center", fontsize=8)
ax.set_yticks(y)
ax.set_yticklabels(board["attack"])
ax.set_xlim(0, 1.2)
ax.set_xlabel("share of targets")
ax.set_title("Same city network: where each attack succeeds")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()
plt.show()

**Reading the chart.** Every attack has a foothold on today’s setup. On our design none succeed. “Reading the wire” is blocked everywhere, but only the QKD links *notice* the tap — post-quantum encryption hides the data without telling you someone is listening. Stolen passwords and GPS spoofing are not quantum problems; we still put them in the design because a secure city network has to stop them.

## Risk score → what we fix first

We reuse the two-clock risk model from notebook 2 on the same systems list.

In [ ]:
data = risk.load()
table, z = risk.run(data, estimate="2025")
print("RSA/ECC break (2025 estimate): median year",
      int(round(np.median(risk.THIS_YEAR + z))),
      "| 1 in 10 chance by", int(round(np.percentile(risk.THIS_YEAR + z, 10))))
print(table[["system", "impact", "risk score", "exposed today, no quantum needed", "must be fixed before"]]
      .to_string(index=False))

## Parts 4–5 — rollout and cost vs security

**Phase 1 (now):** MFA on remote access, protected time, stop plaintext SCADA links, firmware rollback counters. These stop attacks that work today.

**Phase 2:** Post-quantum certificates at the PKI, then ML-KEM on VPNs/ICCP/SCADA and hash-based firmware signatures.

**Phase 3:** QKD on the backbone and the highest-criticality spokes (hospital, water, airport) once fiber and budget allow — for tap *detection*, not instead of PQC.

**Board argument (honest):** A QKD pair costs equipment, dark fiber, and ops; a PQC software/firmware upgrade is cheaper per link. We pay for QKD only where knowing about a tap is worth it. Everywhere else, hybrid ML-KEM + authentication + monitoring is the value. Cost cells in `city_network.json` are still placeholders until we cite vendor prices.

**Working prototype:** BB84 in notebook 1 and the teleportation link in notebook 3. Visuals we built for the pitch: `viz/city_attack_map.html`, `viz/eavesdrop_animation.html`, and `video/secure_the_grid_city_attack.mp4`.